# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR\^2 dataset](https://sen.science/dataset/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. For reproducibility, all data entities (record sets, fields, columns) are referenced by their Croissant `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print metadata, available record sets, fields, and columns by `@id` using mlcroissant's API. All `@id` values are unique to each dataset entity.

In [ ]:
# List available record sets, fields, and columns by their @id
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for f in rs.fields:
            print(f"      Field @id: {f.id}  (name: {f.name}, dataType: {f.data_type})")
            if hasattr(f, 'column') and f.column:
                print(f"        --> Column @id: {f.column.id} (name: {f.column.name})")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> Note: The exact `@id`s will be shown above and used in the extraction below.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records_iterator = dataset.records(record_set=record_set_id)
    records = list(records_iterator)
    df = pd.DataFrame(records)
    # Store df even if empty — user can inspect all record sets
    dataframes[record_set_id] = df
    print(f"\nLoaded {len(df)} records from record set @id: {record_set_id}")
    if len(df.columns) > 0:
        print(f"Fields: {list(df.columns)}")
    else:
        print("No fields loaded.")

## 4. Exploratory Data Analysis (EDA)

We'll select a record set with tabular data, pick a numeric field (referencing by `@id`), then demonstrate filtering, normalization, and grouping.

First, enumerate all available DataFrames and their columns to decide which field and group to use.

In [ ]:
# Preview DataFrame columns and suggest a numeric field for demo
for rsid, df in dataframes.items():
    print(f"\nRecordSet @id: {rsid}")
    print(f"Columns: {df.columns.tolist()}")
    if len(df.columns) > 0 and not df.empty:
        # Try to find numeric columns (int or float) for demo
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if num_cols:
            print(f"  Numeric fields: {num_cols}")

In [ ]:
# For demonstration, pick the first record set with numeric field(s)
numeric_record_set_id = None
numeric_field_id = None
group_field_id = None
for rsid, df in dataframes.items():
    if len(df.columns) > 0 and not df.empty:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if num_cols:
            numeric_record_set_id = rsid
            numeric_field_id = num_cols[0]  # Just pick first numeric col
            # Try to pick a non-numeric for groupby
            non_num_cols = [col for col in df.columns if col not in num_cols]
            if non_num_cols:
                group_field_id = non_num_cols[0]
            break

if numeric_record_set_id is None:
    print("No numeric data found in any record set.")
else:
    print(f"Analysis will use record set: {numeric_record_set_id}")
    print(f"  Numeric field: {numeric_field_id}")
    if group_field_id:
        print(f"  Group field: {group_field_id}")

In [ ]:
# Only proceed if numeric fields are found
if numeric_record_set_id and numeric_field_id:
    df = dataframes[numeric_record_set_id]
    # Filtering on a threshold: pick 10th percentile as demo threshold
    thresh = df[numeric_field_id].quantile(0.10)
    filtered_df = df[df[numeric_field_id] > thresh].copy()
    print(f"Filtered {len(filtered_df)} out of {len(df)} records with '{numeric_field_id}' > {thresh:.3f}")
    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by group_field if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        print(f"\nGrouped by '{group_field_id}' (showing mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric fields to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referenced using their Croissant `@id`s.

Below is an example using matplotlib/seaborn for the selected numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Proceed only if numeric field is available
if numeric_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (from record set {numeric_record_set_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    # If grouping field, show boxplot
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(y=filtered_df[numeric_field_id], x=filtered_df[group_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric records to visualize.")

## 6. Conclusion
This notebook demonstrated how to:
- Load and explore a Croissant dataset by referencing all entities by their `@id` (record sets, fields, columns)
- Retrieve metadata and enumerate structure using `mlcroissant`
- Process tabular data with standard EDA and normalization techniques
- Visualize distributions and grouped data

For further analysis, consider exploring relationships between adoption predictors, intervention outcomes, and socio-demographic features in the context of rangeland management. Always reference dataset elements using their full Croissant `@id` to ensure reproducibility and provenance.